# 数据探索 Notebook

本 notebook 用于探索销售线索数据，了解数据分布、特征统计等。

In [ ]:
import sys
from pathlib import Path

# 添加项目根目录到路径
project_root = Path("..").resolve()
sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from config.config import config
from src.data.loader import DataLoader, check_data_quality

# 设置绘图风格
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False

## 1. 加载数据

In [ ]:
# 加载数据
data_path = config.data.data_path
loader = DataLoader(data_path)
df = loader.load()

print(f"数据量: {len(df)} 行, {len(df.columns)} 列")
df.head()

## 2. 数据质量检查

In [ ]:
# 数据质量报告
quality_report = check_data_quality(df)
print("数据质量报告:")
for key, value in quality_report.items():
    if key != 'missing_values':
        print(f"  {key}: {value}")

In [ ]:
# 缺失值可视化
missing = df.isnull().sum()
missing_pct = missing / len(df) * 100
missing_df = pd.DataFrame({
    'column': missing.index,
    'missing_count': missing.values,
    'missing_pct': missing_pct.values
}).sort_values('missing_pct', ascending=False)

# 显示缺失值超过 10% 的列
high_missing = missing_df[missing_df['missing_pct'] > 10]
if len(high_missing) > 0:
    print(f"缺失值超过 10% 的列 ({len(high_missing)} 个):")
    print(high_missing.head(20))
else:
    print("没有缺失值超过 10% 的列")

## 3. 目标变量分析

In [ ]:
# 目标变量分布
target_cols = [col for col in df.columns if col.startswith('label_')]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, col in enumerate(target_cols[:6]):
    ax = axes[i]
    df[col].value_counts().plot(kind='bar', ax=ax)
    ax.set_title(col)
    ax.set_xlabel('')

plt.tight_layout()
plt.savefig('../outputs/reports/target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 目标变量转化率
print("目标变量转化率:")
for col in target_cols:
    if col in df.columns:
        rate = df[col].mean()
        print(f"  {col}: {rate:.2%}")

## 4. OHAB 评级分布

In [ ]:
# OHAB 评级分布
if 'clue_level' in df.columns:
    print("OHAB 评级分布:")
    print(df['clue_level'].value_counts())
    print()
    print("OHAB 评级比例:")
    print(df['clue_level'].value_counts(normalize=True))
    
    # 可视化
    fig, ax = plt.subplots(figsize=(8, 5))
    df['clue_level'].value_counts().plot(kind='bar', ax=ax)
    ax.set_title('OHAB 评级分布')
    plt.tight_layout()
    plt.savefig('../outputs/reports/ohab_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()

## 5. 特征分布分析

In [ ]:
# 渠道分布
channel_cols = [col for col in df.columns if 'channel' in col.lower()]
print(f"渠道相关列: {channel_cols}")

if 'level_1_channel_name' in df.columns:
    print("\n一级渠道分布:")
    print(df['level_1_channel_name'].value_counts())

In [ ]:
# 数值特征统计
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print(f"数值特征数量: {len(numeric_cols)}")

# 基本统计
df[numeric_cols].describe().T

## 6. 相关性分析

In [ ]:
# 目标变量与数值特征的相关性
target = 'label_arrive_14d'
if target in df.columns:
    correlations = df[numeric_cols + [target]].corr()[target].drop(target)
    correlations = correlations.abs().sort_values(ascending=False)
    
    print(f"与 {target} 相关性最高的特征 (Top 15):")
    print(correlations.head(15))

## 7. 下一步

1. 运行 `python scripts/train_arrive.py` 训练到店预测模型
2. 运行 `python scripts/train_ohab.py` 训练 OHAB 评级模型
3. 查看评估报告和 Top-K 名单